# PlanTo3D — train the segmenter on CubiCasa5K

Trains a U-Net with a ResNet34 encoder to label floor plan pixels as
background, wall, room, door or window. Everything else in PlanTo3D runs on
CPU; this is the one stage that needs a GPU.

**Set the runtime to GPU first:** Runtime → Change runtime type → T4 GPU.

The order below matters. Sections 1–4 set up and then *verify the annotation
parsing against real data* before section 5 spends GPU hours on it. The
parser is unit-tested against synthetic SVG only, so a real sample is the
first genuine check that CubiCasa's 80+ categories collapse to our five
correctly. A wrong mapping trains a model that looks broken for reasons that
have nothing to do with training.

## 1. Confirm the GPU

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("NO GPU. Runtime -> Change runtime type -> T4 GPU, then rerun.")

## 2. Install dependencies and fetch the code

In [ ]:
!pip install -q segmentation-models-pytorch kagglehub opencv-python-headless

In [ ]:
import sys
from pathlib import Path

REPO_URL = "https://github.com/priyanshsoni096-blip/PlanTo3D.git"
repo = Path("/content/PlanTo3D")

if repo.exists():
    !cd {repo} && git pull --quiet
else:
    !git clone --quiet {REPO_URL} {repo}

if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from planto3d.classes import CLASS_NAMES

print(f"classes: {CLASS_NAMES}")

## 3. Download CubiCasa5K

About 6 GB, pulled straight into the Colab VM rather than uploaded from your
machine. `kagglehub` will ask you to authenticate on first use.

In [ ]:
import kagglehub

download_root = Path(kagglehub.dataset_download("qmarva/cubicasa5k"))
print(f"downloaded to {download_root}")

# The archive nests a second cubicasa5k folder; the split files mark the root
# the sample paths in train.txt are relative to.
candidates = [p.parent for p in download_root.rglob("train.txt")]
if not candidates:
    raise SystemExit(f"no train.txt found under {download_root}")

DATA_ROOT = candidates[0]
print(f"data root: {DATA_ROOT}")
print(f"contents:  {sorted(p.name for p in DATA_ROOT.iterdir())[:10]}")

In [ ]:
from planto3d.cubicasa import sample_paths

splits = {
    name: sample_paths(DATA_ROOT, DATA_ROOT / f"{name}.txt")
    for name in ("train", "val", "test")
}
for name, pairs in splits.items():
    print(f"{name:6} {len(pairs):5d} samples")

## 4. Verify the annotation mapping on real data

**Do not skip this.** Everything downstream assumes CubiCasa's categories
collapse correctly onto our five classes. Check that walls trace walls, rooms
fill rooms, and doors and windows appear at openings — if the overlay looks
wrong, fix `planto3d/cubicasa.py` before training rather than after.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

from planto3d.cubicasa import class_distribution, svg_to_mask

# Distinct colours per class: background, wall, room, door, window.
PALETTE = np.array(
    [[255, 255, 255], [40, 40, 40], [150, 200, 255], [255, 80, 80], [80, 220, 120]],
    dtype=np.uint8,
)


def show(pairs, count=3):
    fig, axes = plt.subplots(count, 3, figsize=(15, 5 * count))
    axes = np.atleast_2d(axes)

    for row, (image_path, svg_path) in enumerate(pairs[:count]):
        image = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)
        mask = svg_to_mask(svg_path, image.shape[:2])
        coloured = PALETTE[mask]

        axes[row][0].imshow(image)
        axes[row][0].set_title(image_path.parent.name)
        axes[row][1].imshow(coloured)
        axes[row][1].set_title("mask: wall=black room=blue door=red window=green")
        axes[row][2].imshow(cv2.addWeighted(image, 0.55, coloured, 0.45, 0))
        axes[row][2].set_title("overlay")
        for ax in axes[row]:
            ax.axis("off")

        shares = class_distribution(mask)
        print(
            image_path.parent.name,
            " ".join(f"{CLASS_NAMES[i]}={shares[i]:.1%}" for i in range(5)),
        )

    plt.tight_layout()
    plt.show()


# Architectural sheets resemble the Soni Residence drawings most closely.
architectural = [p for p in splits["train"] if "architectural" in str(p[0])]
print(f"{len(architectural)} architectural samples\n")
show(architectural or splits["train"])

In [ ]:
# Class balance across a sample of the training set. Doors and windows are
# expected to be a fraction of a percent each -- that imbalance is why the
# loss pairs cross-entropy with Dice and the metrics average per class.
import random

random.seed(0)
totals = np.zeros(5)
checked = 0

for image_path, svg_path in random.sample(splits["train"], min(60, len(splits["train"]))):
    image = cv2.imread(str(image_path))
    if image is None:
        continue
    shares = class_distribution(svg_to_mask(svg_path, image.shape[:2]))
    totals += np.array([shares[i] for i in range(5)])
    checked += 1

print(f"mean class share over {checked} samples:")
for index, share in enumerate(totals / max(checked, 1)):
    print(f"  {CLASS_NAMES[index]:11} {share:7.2%}")

if (totals / max(checked, 1))[1] < 0.005:
    print("\nWARNING: almost no wall pixels. The mapping is probably wrong -- check section 4.")

## 5. Smoke run

Two epochs over a handful of samples. Proves the loop runs on this data
before committing to the full job — cheaper to fail here than an hour in.

In [ ]:
import logging

from training.train import train

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", force=True)

train(
    DATA_ROOT,
    "/content/smoke.pt",
    epochs=2,
    batch_size=4,
    size=256,
    limit=24,
    num_workers=2,
)

## 6. Full training run

Roughly 2–4 hours on a T4 for the whole training split. Drop `epochs` or set
`limit` if the session is likely to be cut short — Colab reclaims free
runtimes, and the checkpoint only survives if it reaches Drive (section 7).

The best checkpoint by validation Dice is saved each time it improves, so an
interrupted run still leaves something usable behind.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

CHECKPOINT = Path("/content/drive/MyDrive/planto3d/unet_cubicasa.pt")
CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
print(f"checkpoint will be written to {CHECKPOINT}")

In [ ]:
train(
    DATA_ROOT,
    CHECKPOINT,
    epochs=12,
    batch_size=8,
    learning_rate=3e-4,
    size=512,
    num_workers=2,
)

## 7. Score the held-out test split

In [ ]:
from torch.utils.data import DataLoader

from training.dataset import CubiCasaDataset
from training.train import build_model, evaluate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
state = torch.load(CHECKPOINT, map_location=device, weights_only=False)

model = build_model().to(device)
model.load_state_dict(state["model_state"])

test_loader = DataLoader(
    CubiCasaDataset(DATA_ROOT, DATA_ROOT / "test.txt", size=state["size"]),
    batch_size=8,
    num_workers=2,
)

metrics = evaluate(model, test_loader, device)
print(f"test Dice {metrics['dice']:.4f}   test IoU {metrics['iou']:.4f}\n")
for name, value in metrics["per_class_iou"].items():
    print(f"  {name:11} IoU {value if value is None else round(value, 4)}")

## 8. Try it on the real floor plan

The point of the whole exercise: does the trained model beat the classical
baseline on a drawing style it never saw in training? Upload a page image
exported by `scripts/crop_pages.py`.

In [ ]:
from google.colab import files

from planto3d.classical import classical_mask
from training.dataset import IMAGENET_MEAN, IMAGENET_STD

uploaded = files.upload()
page = cv2.cvtColor(cv2.imread(next(iter(uploaded))), cv2.COLOR_BGR2RGB)

size = state["size"]
resized = cv2.resize(page, (size, size), interpolation=cv2.INTER_AREA)
normalized = (resized.astype(np.float32) / 255.0 - IMAGENET_MEAN) / IMAGENET_STD
batch = torch.from_numpy(normalized).permute(2, 0, 1).unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
    predicted = model(batch).argmax(dim=1)[0].cpu().numpy()

# Back to the page's own resolution so it can be compared with the baseline.
predicted = cv2.resize(
    predicted.astype(np.uint8), (page.shape[1], page.shape[0]), interpolation=cv2.INTER_NEAREST
)
baseline = classical_mask(cv2.cvtColor(page, cv2.COLOR_RGB2BGR))

fig, axes = plt.subplots(1, 3, figsize=(21, 7))
for ax, content, title in zip(
    axes,
    [page, PALETTE[baseline], PALETTE[predicted]],
    ["floor plan", "classical baseline", "trained U-Net"],
):
    ax.imshow(content)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 9. Bring the checkpoint home

About 98 MB. Save it under `models/` in the project, then pass
`--checkpoint` to the pipeline to use the model in place of the baseline.

In [ ]:
files.download(str(CHECKPOINT))